## Motivation

**The Scenario:**
You write a script or notebook. It runs perfectly. You send it to a colleague. It fails immediately.

**Why does this happen?**
1.  **Hidden State:** Variables defined 10 cells ago that you forgot about.
2.  **Dependency Hell:** You possess a library version they don't have.
3.  **Pathing Issues:** Hardcoded paths (`C:/Users/Me/Downloads/...`).

**The Goal:**
Transform a fragile analysis script or notebook into a robust, shareable tool.

# Environment management

## Defining a reproducible environment

- Instead of manually installing packages, we define them in a file called `environment.yml`.
- This is the **recipe** for your environment, which you can manage with **conda** or **mamba**.
- **Conda** is a package manager *and* an environment manager.
- **Mamba** is a reimplementation of Conda in C++; it does the exact same thing, just much faster.

### Why not just `pip install`?
`pip` installs Python packages globally or locally, but it doesn't handle **system-level dependencies**.

## The content of the environment.yml file

```yml
name: risk_analysis_env
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pandas=2.1.0
  - pydantic=2.4.0
  - pip
```

- `name`: name of the environment
- `channels`: channels from where to the dependencies can be optained
- `dependencies`: top-level (not derived) dependencies

## Managing the environment

-  **Create:**
    ```bash
    mamba env create -f environment.yml
    ```
-  **Activate:**
    ```bash
    mamba activate risk_analysis_env
    ```
-  **Update (if you edit the yaml):**
    ```bash
    mamba env update -f environment.yml --prune
    ```

# From messy to maintainable code

## Example: a "messy" risk score script

In [16]:
import pandas as pd

# Hardcoded path - breaks on anyone else's machine
# data = pd.read_csv("C:/Users/JDOE/data_dump_final_v2.csv") 

# Simulating the data load for this notebook
data = [
    [1, "John", 45, "M", 80.5, 30],
    [2, "Jane", 22, "F", 60.0, 25]
]

results = []

for row in data:
    # Magic numbers & Index based access
    if row[2] > 18:
        score = (row[4] * 0.5) + (row[5] * 1.2)
        if score > 50:
            results.append((row[0], score, "High"))
        else:
            results.append((row[0], score, "Low"))

print(results)

[(1, 76.25, 'High'), (2, 60.0, 'High')]


## Problems with this script

- **Fragility:** Relies on list indices, but what if the column order changes?
- **Obscurity:** What are the "magic numbers" `0.5` and `1.2`?
  ```python
  if row[2] > 18:
        score = (row[4] * 0.5) + (row[5] * 1.2)
  ```
- **Inflexibility:** Hardcoded file path.
  ```python
  # Hardcoded path - breaks on anyone else's machine
  # data = pd.read_csv("C:/Users/JDOE/data_dump_final_v2.csv") 
  ```    

## Type hinting, docstrings, constants

- **Type hinting:** Python is dynamic, but hints let your IDE catch bugs (e.g., trying to divide a string).
- **Docstrings (Google Style):** Standardized documentation that explains *arguments*, *returns*, and *raises*.
- **Constants:** Giving names to magic numbers makes the math self-explanatory.

## Example: type hinting, docstrings, constants

In [14]:
from typing import List, Tuple, Literal

# Constants: Centralized configuration
WEIGHT_FACTOR = 0.5
AGE_FACTOR = 1.2
RISK_THRESHOLD = 50.0

# Type Alias for clarity
RiskLevel = Literal["High", "Low"]

def calculate_risk_score(weight: float, age: int) -> float:
    """Calculates the health risk score based on weight and age.

    The formula applies specific weighting factors to prioritize 
    age over weight in the risk assessment.

    Args:
        weight (float): The patient's weight in kg.
        age (int): The patient's age in years.

    Returns:
        float: A calculated risk score (0.0 to 100.0).
    """
    return (weight * WEIGHT_FACTOR) + (age * AGE_FACTOR)

def classify_risk(score: float) -> RiskLevel:
    """Classifies a raw score into a categorical risk level."""
    return "High" if score > RISK_THRESHOLD else "Low"

## Data validation with Pydantic

- Type hints are ignored at runtime.
- If you pass a string `"50"` to a math function, Python might crash or behave unexpectedly.
- **Pydantic** enforces data integrity by parsing and validating data.
- If your input data is messy, the program crashes *early* and with a clear error message.

## Example: Pydantic upgrade for the risk score code

In [17]:
from pydantic import BaseModel, Field, field_validator

class PatientRecord(BaseModel):
    id: int
    name: str
    # Pydantic enforces logic constraints
    age: int = Field(..., gt=0, le=120, description="Age in years")
    weight: float = Field(..., gt=0, description="Weight in kg")
    
    @field_validator('name')
    @classmethod
    def name_must_be_capitalized(cls, v: str) -> str:
        if not v[0].isupper():
            raise ValueError('Name must start with a capital letter')
        return v

# --- Simulation ---
print("--- Testing Validation ---")
try:
    # This will fail because age is negative
    bad_patient = PatientRecord(id=1, name="john", age=-5, weight=70)
except Exception as e:
    print(f"❌ Validation Error:\n{e}")

# This works and converts types automatically if safe to do so
good_patient = PatientRecord(id=2, name="Alice", age=30, weight=65.5)
print(f"✅ Valid Patient: {good_patient}")
print(f"   Accessed property: {good_patient.age}")

--- Testing Validation ---
❌ Validation Error:
2 validation errors for PatientRecord
name
  Value error, Name must start with a capital letter [type=value_error, input_value='john', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
age
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
✅ Valid Patient: id=2 name='Alice' age=30 weight=65.5
   Accessed property: 30


# 3. Adding a command line interface (CLI)

## Writing CLIs with `argparse`

- To make code reproducible, we must separate **logic** from **execution**.
- We shouldn't have to open a Python file to change a parameter (like the input file name).
- **argparse** allows us to pass arguments from the terminal.
- `--flag`: Optional parameters (usually with defaults).
- `positional`: Mandatory arguments (like input files).
- `help`: Auto-generated documentation.

## Example: a CLI for our risk score

In [18]:
import argparse

def parse_arguments(args_list=None):
    """
    Parses command line arguments.
    
    Args:
        args_list (list): Allow passing arguments manually for Notebook testing.
                          In a real script, we default to None (sys.argv).
    """
    parser = argparse.ArgumentParser(
        description="Process patient data to calculate risk scores."
    )

    # Positional argument (Required)
    parser.add_argument(
        "input_file",
        type=str,
        help="Path to the input CSV file."
    )
    
    # Optional Flag with default
    parser.add_argument(
        "--threshold", "-t",
        type=float,
        default=50.0,
        help="Risk score threshold override (default: 50.0)."
    )

    # In a real script: return parser.parse_args()
    return parser.parse_args(args_list)

# Simulating a terminal call: 
# python script.py data.csv --threshold 60
args = parse_arguments(["data.csv", "--threshold", "60"])

print(f"📂 Input: {args.input_file}")
print(f"⚙️ Threshold: {args.threshold}")

📂 Input: data.csv
⚙️ Threshold: 60.0


## Putting it all together

In [19]:
import argparse
from typing import List, Tuple, Literal, Optional
from pydantic import BaseModel, Field, field_validator

# --- 1. CONFIGURATION / CONSTANTS ---
WEIGHT_FACTOR = 0.5
AGE_FACTOR = 1.2
RISK_THRESHOLD = 50.0

# --- 2. DATA MODELS (VALIDATION) ---
class PatientRecord(BaseModel):
    """Schema for incoming patient data with strict validation."""
    id: int
    name: str
    age: int = Field(..., gt=0, le=120)
    weight: float = Field(..., gt=0)
    
    @field_validator('name')
    @classmethod
    def name_must_be_capitalized(cls, v: str) -> str:
        if not v[0].isupper():
            raise ValueError('Name must start with a capital letter')
        return v

# --- 3. BUSINESS LOGIC ---
RiskLevel = Literal["High", "Low"]

def calculate_risk_score(weight: float, age: int) -> float:
    """Calculates the health risk score based on weight and age."""
    return (weight * WEIGHT_FACTOR) + (age * AGE_FACTOR)

def classify_risk(score: float, threshold: float = RISK_THRESHOLD) -> RiskLevel:
    """Classifies a raw score into a categorical risk level."""
    return "High" if score > threshold else "Low"

# --- 4. EXECUTION CORE ---
def run_analysis(input_path: str, threshold: float):
    """
    The main controller that orchestrates the data pipeline.
    In a real app, this would use pd.read_csv(input_path).
    """
    print(f"--- Processing File: {input_path} ---")
    
    # Mock data source
    raw_data = [
        {"id": 1, "name": "Alice", "age": 30, "weight": 60.5},
        {"id": 2, "name": "Bob", "age": 45, "weight": 85.0},
        {"id": 3, "name": "charlie", "age": -5, "weight": 70} # This will fail validation
    ]
    
    processed_results = []
    
    for item in raw_data:
        try:
            # Step A: Validate
            patient = PatientRecord(**item)
            
            # Step B: Logic
            score = calculate_risk_score(patient.weight, patient.age)
            risk = classify_risk(score, threshold)
            
            processed_results.append({
                "patient": patient.name,
                "score": round(score, 2),
                "risk": risk
            })
            
        except (ValueError, TypeError) as e:
            print(f"⚠️ Skipping ID {item.get('id')}: Invalid data provided. ({e})")

    return processed_results

# --- 5. INTERFACE (CLI) ---
def main():
    parser = argparse.ArgumentParser(description="Professional Risk Analysis Tool")
    
    parser.add_argument("input", type=str, help="Path to input data")
    parser.add_argument("--threshold", "-t", type=float, default=50.0, help="Risk threshold")

    # For Jupyter/Quarto demonstration, we mock the args:
    args = parser.parse_args(["patients.csv", "--threshold", "55.0"])
    
    results = run_analysis(args.input, args.threshold)
    
    print("\n--- Final Analysis Results ---")
    for r in results:
        print(r)

if __name__ == "__main__":
    main()

--- Processing File: patients.csv ---
⚠️ Skipping ID 3: Invalid data provided. (2 validation errors for PatientRecord
name
  Value error, Name must start with a capital letter [type=value_error, input_value='charlie', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error
age
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than)

--- Final Analysis Results ---
{'patient': 'Alice', 'score': 66.25, 'risk': 'High'}
{'patient': 'Bob', 'score': 96.5, 'risk': 'High'}


# Python scripts and projects

## How to run Python scripts

- Once you move this code out of Jupyter and into a file (e.g., `risk_analyzer.py`), how do you actually run it?
- **The standard call:**
```bash
python risk_analyzer.py data/patients.csv --threshold 75
```
- **The `__main__` block:**
```python
if __name__ == "__main__":
    args = parse_arguments()
    main(args)
```
- Ensures code only runs when the script is executed directly, not when imported by another tool.

## The "src" layout for Pythin packaging

- As your project grows, keeping everything in one file becomes unmanageable.
- You need a standard structure.
- This is the modern standard for Python packaging.

```text
my_project/
├── environment.yml      # The Mamba/Conda spec
├── pyproject.toml       # Build tools & metadata
├── README.md            # Documentation
├── src/                 # Source code lives here
│   └── risk_pkg/        # The actual package name
│       ├── __init__.py  # Marks directory as package
│       ├── analysis.py  # Logic
│       └── cli.py       # Argparse entry point
└── tests/               # Pytest files
    └── test_analysis.py
```

## Why `__init__.py`?

- The `__init__.py` file signals to Python that a directory is an importable package.
- It can be empty, or it can expose specific functions to make imports cleaner.
- **Without `__init__.py` in `src/risk_pkg/`:**
 ```python
 from src.risk_pkg.analysis import calculate_risk_score
 ```
- **With `__init__.py` importing `calculate_risk_score`:**
- Content of `__init__.py`:
  ```python
  from .analysis import calculate_risk_score
  ```
- Usage:
  ```python
  from risk_pkg import calculate_risk_score
  ```

## The `README.md` file

- Your project's front door
- Provides the **human-readable context** that the code cannot.

### The standard "must-haves"
-  **Project title & description:** What does this do and why?
-  **Installation:** How do I set up the Mamba environment?
-  **Usage:** Show me the exact command to run the script.
-  **Testing:** How do I verify it's working?
-  **License:** Can I actually use this?

# Automated testing with Pytest

## Why Pytest?
- No need to wrap tests in classes.
- Automatically finds files named `test_*.py`.
- Provides detailed diffs of what went wrong.

### The testing mindset
- **Happy path:** Does it work with perfect data?
- **Edge cases:** Does it work at the boundaries (e.g., age = 120)?
- **Expected failures:** Does it throw an error when given garbage data?

## Structuring the `tests/` Folder

- Following our **src layout**, your tests live outside the source code.
- This ensures they aren't packaged with your code but are easily accessible during development.

```text
my_project/
├── src/
│   └── risk_pkg/
│       ├── analysis.py
│       └── ...
└── tests/
    ├── __init__.py
    ├── test_analysis.py    # Tests for our math logic
    └── test_models.py      # Tests for Pydantic validation
```

## Contents of `tests/test_analysis.py`

In [20]:
import pytest
# In a real project: from risk_pkg.analysis import calculate_risk_score
# For this notebook, we use the functions defined in previous cells

def test_calculate_risk_score_valid():
    """Test the math logic with standard values."""
    # weight=80, age=30 -> (80 * 0.5) + (30 * 1.2) = 40 + 36 = 76
    result = calculate_risk_score(80.0, 30)
    assert result == 76.0

def test_classify_risk_thresholds():
    """Test that the classification hits the boundary correctly."""
    assert classify_risk(49.9, threshold=50) == "Low"
    assert classify_risk(50.1, threshold=50) == "High"

def test_patient_validation_error():
    """Test that Pydantic catches invalid data."""
    with pytest.raises(ValueError):
        # Age exceeds the 'le=120' constraint
        PatientRecord(id=1, name="Old Person", age=150, weight=70)

def test_patient_name_capitalization():
    """Test our custom validator for name capitalization."""
    with pytest.raises(ValueError, match="Name must start with a capital letter"):
        PatientRecord(id=1, name="bob", age=30, weight=70)

## Contents of `tests/test_models.py`

In [21]:
import pytest
# In a real project: from risk_pkg.analysis import PatientRecord
# For this notebook, we use the classes defined in previous cells

def test_patient_validation_error():
    """Test that Pydantic catches invalid data."""
    with pytest.raises(ValueError):
        # Age exceeds the 'le=120' constraint
        PatientRecord(id=1, name="Old Person", age=150, weight=70)

def test_patient_name_capitalization():
    """Test our custom validator for name capitalization."""
    with pytest.raises(ValueError, match="Name must start with a capital letter"):
        PatientRecord(id=1, name="bob", age=30, weight=70)

## How to run your tests

- Once your tests are written, you execute them from the **root** of your project directory using the terminal.

### Basic command
```bash
pytest tests/
```

### Advanced flags
- `-v` (verbose): Shows the name of every test being run.
- `-x` (exit on first fail): Stops the suite immediately when a test fails.
- `--cov` (coverage): If the pytest-cov plugin is installed, it shows you which lines of your code are not yet tested.

## Updated `environment.yml`

```yml
name: risk_analysis_env
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pandas=2.1.0
  - pydantic=2.4.0
  # Testing suite
  - pytest
  - pytest-cov
  - pip
```

# Summary and exercises

## Summary

1.  **Environment:** Always use `environment.yml` with pinned versions.
2.  **Documentation:** Use Docstrings and Type Hints.
3.  **Validation:** Trust no input—use Pydantic.
4.  **CLI:** Use `argparse` to parameterize your scripts.
5.  **Structure:** Separate source code (`src/`) from tests and config.
6.  **Tests:** Write unit tests for your source code.

## Exercises

- Generate a `risk_pkg` Python package, using the `src` file structure outlined above.
- Distribute the code from the previous cells into the individual files and run the tests.
- Write a README with all the "must-haves" mentioned above.